# SequenceVis

Converts a CSV matrix of position-specific amino acid frequencies into foreground/background sequence data for visualization.

## Overview

This notebook takes a CSV table of amino acid probabilities at positions -5 to +4 (relative to a motif center) and:
1. **Foreground (fg)**: Converts probabilities into a discrete pool of sequences enriched for the motif
2. **Background (bg)**: Generates random sequences with a constraint at position 0 (only S or T)

## Workflow
1. Define the amino acid alphabet
2. Generate a random input CSV (or upload your own)
3. Convert the CSV into foreground/background datasets
4. Export results as `fg.csv` and `bg.csv`

## Setup

Install required dependencies and import libraries.

In [ ]:
# Install required packages
!pip install pandas numpy -q

In [ ]:
import pandas as pd
import numpy as np

## 1. Amino Acid Alphabet

Define the standard 20-amino-acid alphabet used throughout the pipeline.

In [ ]:
# amino alphabet
AMINO_ALPH = ["A","C","D","E","F","G","H","I","K","L","M","N","P","Q","R","S","T","V","W","Y"]
AMINOS = 20

## 2. Generate Random Input CSV

Creates a synthetic 20×9 matrix (20 amino acids × 9 positions: -5 through +3) with random values normalized so each column sums to 20 (i.e., probabilities out of 20).

> **Note**: If you have your own CSV file, skip this cell and upload your file instead. The expected format is a CSV with a `titles` column (amino acid one-letter codes) and numeric columns for each position.

In [ ]:
cols = list(range(-5, 5))  # -5 through 4

rng = np.random.default_rng(seed=None)

# Generate a 20x10 matrix all at once
raw = rng.uniform(1, 100, size=(len(AMINO_ALPH), len(cols)))
normalized = (raw / raw.sum(axis=0)) * 20

# Create DataFrame
df = pd.DataFrame(normalized, columns=cols)
df.insert(0, 'titles', AMINO_ALPH)

df.to_csv('input.csv', index=False)
print("input.csv generated successfully")
df.head()

### (Optional) Upload Your Own CSV

If you have a custom CSV file, run the cell below to upload it. It will overwrite the generated `input.csv`.

In [ ]:
from google.colab import files

uploaded = files.upload()
for fn in uploaded.keys():
  print(f'Uploaded file: {fn}')
  # If the uploaded file should be used as input, rename/move it
  # For example, if you upload 'my_data.csv', copy it to 'input.csv':
  # import shutil
  # shutil.copy(fn, 'input.csv')

## 3. CSV to Foreground/Background Converter

This is the core conversion logic:

- **Foreground**: Each position's amino acid probabilities are converted into a discrete pool of `FG_SIZE` amino acids. For example, if R has probability 4/20 (20%) at position -5, then 200 out of 1000 items in column -5 become R.
- **Background**: Random sequences of length 10 are generated, with position 0 (index 5) constrained to only S or T.

In [ ]:
# Constants
PATH = "input.csv"
SIGFIGS = 2
SEED = 42
TRUERAND = True  # enable truly random seed

rng = np.random.default_rng(seed=None if TRUERAND else SEED)

# Size constants
FG_SIZE = 1000
BG_SIZE = 5000
CHAIN_LENGTH = 10

In [ ]:
def norm(x, value):
    return (x / AMINOS) * value

def fill_blanks(df):
    """
    Replaces missing values (NaN/None) and empty/whitespace-only string cells with 0.
    """
    return df.replace(r'^\s*$', 0, regex=True).fillna(0)

def open_csv():
    df = pd.read_csv(PATH)
    return fill_blanks(df)

def normalize(subset, val):
    return (subset / AMINOS) * val

def csv_to_quantity():
    df = open_csv()
    numeric_cols = df.select_dtypes(include=[np.number]).columns

    df_normalized = df.copy()
    df_normalized[numeric_cols] = normalize(df[numeric_cols], FG_SIZE)

    return df_normalized.round(SIGFIGS)

In [ ]:
def fg():
    quant = csv_to_quantity()

    # Determine amino acid labels from a 'titles' column or default alphabet
    if "titles" in quant.columns:
        amino_acids = quant["titles"].values
    else:
        amino_acids = AMINO_ALPH

    # Identify position columns (e.g., -5 to 4 or string representations '-5' to '4')
    pos_cols = [col for col in quant.columns if str(col).lstrip("-").isdigit()]

    fg_dict = {}

    for col in pos_cols:
        # Convert quantities to integer row counts
        counts = quant[col].round().astype(int)

        # Build an exact pool of amino acids matching target quantities
        col_pool = []
        for aa, count in zip(amino_acids, counts):
            col_pool.extend([aa] * count)

        # Handle any minor floating point rounding mismatches to ensure exactly FG_SIZE
        if len(col_pool) < FG_SIZE:
            col_pool.extend(
                rng.choice(amino_acids, size=FG_SIZE - len(col_pool))
            )
        elif len(col_pool) > FG_SIZE:
            col_pool = col_pool[:FG_SIZE]

        # Convert to numpy array and shuffle randomly
        col_arr = np.array(col_pool)
        rng.shuffle(col_arr)

        fg_dict[col] = col_arr

    return pd.DataFrame(fg_dict)

In [ ]:
def bg(seq_len=CHAIN_LENGTH):  
    # 1. Generate random matrix for all positions
    char_matrix = rng.choice(AMINO_ALPH, size=(BG_SIZE, seq_len))
    
    # 2. Override position 0 (index 5) to only pick S or T
    pos_zero_idx = 5  
    char_matrix[:, pos_zero_idx] = rng.choice(["S", "T"], size=BG_SIZE)
    
    # 3. Join into strings
    sequences = ["".join(row) for row in char_matrix]
    
    return pd.DataFrame({"bg": sequences})

## 4. Run the Conversion

Execute the foreground and background generators and preview the results.

In [ ]:
fg_dat = fg()
print("////////////// foreground generator tests //////////////")
fg_dat.head()

In [ ]:
bg_dat = bg()
print("\n////////////// background generator tests //////////////")
print(bg_dat["bg"].head(10).tolist(), end="... and etc\n")
print("//////////////       tests  concluded     //////////////\n")

## 5. Export Results

Save the foreground and background data to CSV files and download them.

In [ ]:
bg_dat.to_csv("bg.csv", index=False)
fg_dat.to_csv("fg.csv", index=False)
print("Exported fg.csv and bg.csv")

In [ ]:
from google.colab import files

files.download("fg.csv")
files.download("bg.csv")

## Summary

- **`fg.csv`**: Foreground dataset — each column is a position (-5 to +4), each row is a sampled amino acid. The distribution at each position matches the input probabilities scaled to `FG_SIZE` (1000) rows.
- **`bg.csv`**: Background dataset — 5000 random 10-mer sequences with position 0 constrained to S/T.

These files can be used for downstream motif visualization (e.g., sequence logos, position weight matrices, etc.).